# NAM 与基线模型对比实验

本 Notebook 用于在 Google Colab 上运行 Neural Additive Models (NAM) 与基线模型的对比实验。

**使用步骤：**
1. 上传项目文件夹到 Google Drive
2. 在 Colab 中打开此 Notebook
3. 按顺序运行下面的单元格

## 1️⃣ 挂载 Google Drive

In [ ]:
from google.colab import drive
import os

# 挂载 Google Drive
drive.mount('/content/drive')

print("✓ Google Drive 已挂载")

## 2️⃣ 切换到项目目录

**重要：** 修改下面的路径为你的项目在 Google Drive 中的实际路径

In [ ]:
# 修改这里的路径！
# 例如：如果你的项目在 "我的云端硬盘/neural_additive_models"
PROJECT_PATH = '/content/drive/MyDrive/neural_additive_models'

# 切换到项目目录
os.chdir(PROJECT_PATH)

# 验证路径
print(f"当前目录: {os.getcwd()}")
print("\n项目文件:")
!ls -lh | head -20

## 3️⃣ 安装依赖包

In [ ]:
# 安装基线对比所需的依赖
!pip install -q xgboost scikit-learn interpret pandas numpy

print("✓ 依赖包安装完成")

## 4️⃣ 方案选择

选择你要运行的方案：
- **方案 A**: 使用示例数据快速测试
- **方案 B**: 使用自己的 CSV 数据
- **方案 C**: 使用 NAM 论文数据集（需要从 GCS 下载）

### 方案 A: 使用示例数据（最快）

In [ ]:
# 运行内置示例
!python baseline_models.py

### 方案 B: 使用你的 CSV 数据

In [ ]:
# 上传 CSV 文件
from google.colab import files

print("请上传你的 CSV 数据文件...")
uploaded = files.upload()

# 获取文件名
csv_filename = list(uploaded.keys())[0]
print(f"\n✓ 已上传: {csv_filename}")

In [ ]:
# 运行对比（修改参数）
DATA_PATH = csv_filename
TARGET_COLUMN = 'target'  # 修改为你的目标列名称
TASK = 'classification'   # 或 'regression'

!python run_experiment.py \
    --data_path {DATA_PATH} \
    --target_column {TARGET_COLUMN} \
    --task {TASK}

### 方案 C: 使用 NAM 论文数据集

In [ ]:
# 测试 Breast Cancer 数据集（sklearn 内置）
from sklearn.datasets import load_breast_cancer
import pandas as pd

# 加载数据
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# 保存为 CSV
df.to_csv('breast_cancer.csv', index=False)

print("✓ Breast Cancer 数据集已准备好")
print(f"数据形状: {df.shape}")
print(f"\n前5行:")
df.head()

In [ ]:
# 运行对比
!python run_experiment.py \
    --data_path breast_cancer.csv \
    --target_column target \
    --task classification

## 5️⃣ 查看结果

In [ ]:
# 查看生成的结果文件
!ls -lh comparison_results/

In [ ]:
# 读取并显示结果
import pandas as pd

# 找到最新的结果文件
import glob
result_files = glob.glob('comparison_results/*_comparison.csv')

if result_files:
    latest_result = sorted(result_files)[-1]
    print(f"读取结果: {latest_result}\n")
    
    results_df = pd.read_csv(latest_result)
    print("="*70)
    print("模型对比结果")
    print("="*70)
    print(results_df.to_string(index=False))
else:
    print("未找到结果文件")

In [ ]:
# 可视化结果
import matplotlib.pyplot as plt

if result_files:
    results_df = pd.read_csv(latest_result)
    
    # 提取测试集性能指标
    test_metric_cols = [col for col in results_df.columns if 'Test' in col and ('AUROC' in col or 'RMSE' in col)]
    
    if test_metric_cols:
        metric_col = test_metric_cols[0]
        
        # 绘图
        plt.figure(figsize=(10, 6))
        plt.barh(results_df['Model'], results_df[metric_col])
        plt.xlabel(metric_col)
        plt.title('模型性能对比')
        plt.tight_layout()
        plt.show()
        
        # 训练时间对比
        if 'Train Time (s)' in results_df.columns:
            plt.figure(figsize=(10, 6))
            plt.barh(results_df['Model'], results_df['Train Time (s)'])
            plt.xlabel('训练时间 (秒)')
            plt.title('模型训练时间对比')
            plt.tight_layout()
            plt.show()

## 6️⃣ 下载结果

In [ ]:
# 打包结果文件
!zip -r comparison_results.zip comparison_results/

# 下载到本地
from google.colab import files
files.download('comparison_results.zip')

print("✓ 结果已打包并开始下载")

## 🔧 高级用法

### 只运行特定模型

In [ ]:
# 只运行 XGBoost 和 CART
!python run_experiment.py \
    --data_path breast_cancer.csv \
    --target_column target \
    --task classification \
    --models xgboost cart

### 使用 Python API

In [ ]:
from baseline_comparison import BaselineComparison
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 加载数据
data = load_breast_cancer()
X, y = data.data, data.target

# 分割数据
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# 标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 运行对比
comparison = BaselineComparison(regression=False, random_state=42)
results_df = comparison.train_and_evaluate(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test
)

print(results_df)

## 📚 帮助文档

查看更多使用说明：

In [ ]:
# 查看快速开始指南
!cat COMPARISON_QUICK_START.md

In [ ]:
# 查看数据集说明
!cat DATASETS.md

## 💡 常见问题

### Q: 如何修改项目路径？
在第2个单元格中修改 `PROJECT_PATH` 变量。

### Q: 如何使用自己的数据？
使用方案 B，上传 CSV 文件并修改 `TARGET_COLUMN` 和 `TASK`。

### Q: 如何只运行某些模型？
使用 `--models` 参数，例如：`--models xgboost cart mlp`

### Q: 结果保存在哪里？
结果保存在 `comparison_results/` 目录中，包含 CSV 和 Markdown 报告。